In [ ]:
# EXP (no es del baseline): Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API.
# Arma /tmp/lab/input/<nombre> -> carpeta real y se para en /tmp/lab/work, asi '../input/<nombre>/...' del baseline funciona sin tocarlo.
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        patron = "/kaggle/input/" + "*/" * prof
        for d in glob.glob(patron):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(f"{nombre}/{archivo} no aparece en /kaggle/input: {sorted(glob.glob('/kaggle/input/*/*'))[:50]}")
_inputs = {"petfinder-adoption-prediction": "train/train.csv",
           "examen-lab-ii-preprocesado-texto": "train.parquet",
           "examen-lab-ii-preprocesado-img": "train.parquet"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        destino = f"/tmp/lab/input/{n}"
        if not os.path.exists(destino):
            os.symlink(_buscar(n, a), destino)
    os.chdir("/tmp/lab/work")
print({n: os.path.realpath(f"../input/{n}") for n in _inputs})

In [ ]:
# Importación de Librerías
import pandas as pd 

from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from lightgbm import LGBMClassifier, early_stopping

def metric(y_true, y_pred):
    res = cohen_kappa_score(y_true, y_pred.reshape((y_true.shape[0], 5), order="F").argmax(axis=1), weights= 'quadratic')
    return "kappa", res, True

In [ ]:
train = pd.read_csv('../input/petfinder-adoption-prediction/train/train.csv').set_index("PetID")
test = pd.read_csv('../input/petfinder-adoption-prediction/test/test.csv').set_index("PetID")
folds = [train.index[test_idx] for train_idx, test_idx in KFold(5).split(train)]

In [ ]:
for c in ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2',
       'Color3', 'FurLength', 'MaturitySize', 'Vaccinated', 'Dewormed',
       'Sterilized', 'Health', 'State', 'RescuerID']:
    train[c] = train[c].astype("category")    
    test[c] = test[c].astype("category")

In [ ]:
train = train\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/train.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/train.parquet"))\
    .select_dtypes(exclude="O")
train

In [ ]:
test = test\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/test.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/test.parquet"))\
    .select_dtypes(exclude="O")
test

In [ ]:
fold_preds = []
test_preds = []
for i, fold in enumerate(folds):
    X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_train = train.drop(fold)["AdoptionSpeed"]

    X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_valid = train.loc[fold, "AdoptionSpeed"]

    learner = LGBMClassifier()
    learner.fit(X_train, y_train, )
    
    learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                eval_metric= metric, callbacks=[early_stopping(5)])
    fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name="AdoptionSpeed"))
    test_preds.append(pd.DataFrame(learner.predict_proba(test[X_train.columns]), index=test.index))

fold_preds = pd.concat(fold_preds)
test_preds = sum(test_preds).idxmax(axis=1).rename("AdoptionSpeed")

In [ ]:
# EXP-012: baseline ya corrido arriba; ahora la variante con el sentiment del documento completo
import json
import numpy as np
pred_base = fold_preds

def sentiment_documento(pid, carpeta):
    f = f"../input/petfinder-adoption-prediction/{carpeta}/{pid}.json"
    if not os.path.exists(f):
        return {}
    doc = json.load(open(f, encoding="utf-8")).get("documentSentiment", {})
    return {"sentdoc_score": doc.get("score", np.nan), "sentdoc_magnitude": doc.get("magnitude", np.nan)}

for df, carpeta in ((train, "train_sentiment"), (test, "test_sentiment")):
    nuevas = pd.DataFrame.from_dict({pid: sentiment_documento(pid, carpeta) for pid in df.index}, orient="index").reindex(df.index)
    for c in nuevas.columns:
        df[c] = nuevas[c].astype(float)
COLS_NUEVAS = [c for c in train.columns if c.startswith("sentdoc_")]
print(COLS_NUEVAS, "filas sin sentiment en train:", int(train["sentdoc_score"].isna().sum()))

In [ ]:
def correr(train, test):
    fold_preds = []
    test_preds = []
    importancias = []
    for i, fold in enumerate(folds):
        X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
        y_train = train.drop(fold)["AdoptionSpeed"]

        X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
        y_valid = train.loc[fold, "AdoptionSpeed"]

        learner = LGBMClassifier()
        learner.fit(X_train, y_train, )
        
        learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                    eval_metric= metric, callbacks=[early_stopping(5)])
        fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name="AdoptionSpeed"))
        test_preds.append(pd.DataFrame(learner.predict_proba(test[X_train.columns]), index=test.index))
        importancias.append(pd.Series(learner.feature_importances_, index=X_train.columns))

    fold_preds = pd.concat(fold_preds)
    test_preds = sum(test_preds).idxmax(axis=1).rename("AdoptionSpeed")
    return fold_preds, test_preds, importancias

In [ ]:
pred_exp, test_preds, imp_exp = correr(train, test)

In [ ]:
def qwk_por_fold(fold_preds):
    return [cohen_kappa_score(train.loc[f, "AdoptionSpeed"], fold_preds.loc[f], weights="quadratic") for f in folds]

res = pd.DataFrame({"baseline": qwk_por_fold(pred_base), "exp_012": qwk_por_fold(pred_exp)},
                   index=[f"fold_{i}" for i in range(len(folds))])
res["delta"] = res["exp_012"] - res["baseline"]
print(res.round(4).to_string())
print("media baseline", round(res["baseline"].mean(), 4), "desvio", round(res["baseline"].std(), 4))
print("media exp_012 ", round(res["exp_012"].mean(), 4), "desvio", round(res["exp_012"].std(), 4))
print("delta media   ", round(res["delta"].mean(), 4))
print("folds que mejoran:", int((res["delta"] > 0).sum()), "de", len(folds))
print("importancia (split) por fold de las columnas nuevas:")
print(pd.DataFrame([s[COLS_NUEVAS] for s in imp_exp], index=[f"fold_{i}" for i in range(len(folds))]).to_string())

In [ ]:
test_preds.to_csv("submission.csv")
!head submission.csv

In [ ]:
# EXP (no es del baseline): si se cambio de carpeta, dejar submission.csv en el output de Kaggle
import shutil
if os.getcwd() != "/kaggle/working" and os.path.exists("submission.csv"):
    shutil.copy("submission.csv", "/kaggle/working/submission.csv")